# Inverse-PINN for Scout Odometry Correction

Colab end-to-end notebook.

Pipeline: upload 2 CSVs (GT + Odom) -> align/split -> build model + physics loss -> train -> plot -> test eval.

Requires `data_module.py`, `model_module.py`, `train_module.py` alongside this notebook (see Cell 2).

## Cell 1: Environment check + minimal installs

In [ ]:
import sys, platform
print('python:', sys.version.split()[0], '|', platform.platform())

# torch / numpy / pandas usually pre-installed on Colab; install only if missing.
try:
    import torch, numpy as np, pandas as pd, matplotlib  # noqa: F401
    print('torch :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
    print('numpy :', np.__version__)
    print('pandas:', pd.__version__)
except Exception as e:
    print('Installing missing deps:', e)
    !pip install -q torch numpy pandas matplotlib

## Cell 2: Import the 3 .py modules

Choose ONE of the two options below (keep the other commented).

- **Option A**: mount Google Drive and point to the folder containing the .py files.
- **Option B**: use `files.upload()` to upload the three files from your local machine.

In [ ]:
import os, sys, importlib

# =====================================================================
# Option A: Google Drive mount
# ---------------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_DIR = '/content/drive/MyDrive/AIR_Labs/2026/클로드실험/experiment_code'
# sys.path.insert(0, CODE_DIR)

# =====================================================================
# Option B: files.upload() — upload data_module.py, model_module.py, train_module.py
# ---------------------------------------------------------------------
from google.colab import files  # type: ignore
print('Upload: data_module.py, model_module.py, train_module.py')
uploaded = files.upload()
CODE_DIR = '/content'
for name, payload in uploaded.items():
    with open(os.path.join(CODE_DIR, name), 'wb') as f:
        f.write(payload)
sys.path.insert(0, CODE_DIR)

# =====================================================================
# Import
import data_module, model_module, train_module
for m in (data_module, model_module, train_module):
    importlib.reload(m)
print('Loaded:', data_module.__name__, model_module.__name__, train_module.__name__)

## Cell 3: Upload the two CSV files (GT + Odom)

In [ ]:
from data_module import upload_data_colab
GT_PATH, ODOM_PATH = upload_data_colab(gt_hint='gt', odom_hint='odom', save_dir='/content')
print('GT  :', GT_PATH)
print('Odom:', ODOM_PATH)

## Cell 4: Align, split, fit scaler, build dataloaders + stationary ratio

In [ ]:
from data_module import (load_and_align, split_timeseries, fit_scaler,
                          build_dataloaders, INPUT_COLS, detect_stationary)

BATCH_SIZE = 64

df = load_and_align(GT_PATH, ODOM_PATH, method='nearest')
print('aligned rows:', len(df))

df_tr, df_va, df_te = split_timeseries(df, ratios=(0.7, 0.15, 0.15))
print(f'split  train={len(df_tr)}  val={len(df_va)}  test={len(df_te)}')

scaler = fit_scaler(df_tr, [f'od_{c}' for c in INPUT_COLS])
loaders = build_dataloaders(df_tr, df_va, df_te, scaler,
                            batch_size=BATCH_SIZE, num_workers=0,
                            shuffle_train=False)

# overall stationary ratio (odom side) on the full aligned df for reporting
_ = detect_stationary(df, thresh_trans=1e-4, thresh_ang=1e-4)

## Cell 5: Build model + physics loss + smoke forward pass

In [ ]:
import torch
from model_module import build_model, build_loss
from train_module import compute_data_weights, TrainConfig, set_seed

set_seed(42)

w_x, w_yaw = compute_data_weights(df_tr)
print(f'data_weights: w_x={w_x:.3e}  w_yaw={w_yaw:.3e}')

model = build_model({'name': 'pinn',
                     'model': {'hidden_dim': 128, 'hidden_blocks': 3, 'dropout': 0.1}})
loss_fn = build_loss({'loss': {
    'w_data': 1.0, 'w_stationary': 1.0, 'w_nonholonomic': 0.1,
    'w_coeff': 0.01, 'w_magnitude': 1e-3, 
    'data_weights': (w_x, w_yaw),
}})

n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params:,}')
print('init coeffs:', model.coefficient_values())

# smoke forward
batch = next(iter(loaders['train']))
with torch.no_grad():
    pred = model(batch['x'])
    total, parts = loss_fn(pred, batch['y'], batch['x_raw'],
                            stationary=batch['stationary'], coeffs=model.coefficients())
print('pred shape:', tuple(pred.shape), '| parts:', {k: float(v) for k, v in parts.items()})

## Cell 6: Train (AdamW + Cosine + early stopping)

In [ ]:
from train_module import train_api

SAVE_DIR = '/content/runs/pinn'

cfg = TrainConfig(
    epochs=300,
    mlp_lr=1e-3,
    coeff_lr_mult=0.3,
    weight_decay=1e-4,
    grad_clip_max_norm=1.0,
    scheduler='cosine',
    early_stop_patience=20,
    seed=42,
    device='auto',
    amp=True,
    deterministic=True,
    log_every=10,
    run_name='pinn_colab',
)

result = train_api(model, loss_fn, loaders['train'], loaders['val'], cfg, save_dir=SAVE_DIR)
print('best_path :', result['best_path'])
print('best_epoch:', result['history'].get('best_epoch'))
print('best_val  :', result['history'].get('best_val_data'))
print('final_coef:', result['history'].get('final_coefficients'))

## Cell 7: Loss curves + coefficient trajectory

In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{SAVE_DIR}/history.json', 'r') as f:
    hist = json.load(f)

rows = hist['epochs']
ep = [r['epoch'] for r in rows]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes[0, 0]
ax.plot(ep, [r['train_total'] for r in rows], label='train_total')
ax.plot(ep, [r['val_total'] for r in rows], label='val_total')
ax.set_title('Total loss'); ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(ep, [r['train_data'] for r in rows], label='train_data')
ax.plot(ep, [r['val_data'] for r in rows], label='val_data')
ax.set_title('Data loss (MSE)'); ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
for k in ('train_stationary', 'train_nonholonomic', 'train_coeff', 'train_magnitude'):
    ax.plot(ep, [r[k] for r in rows], label=k)
ax.set_title('Physics / regularization components (train)')
ax.set_yscale('log'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
coefs = hist['coefficients']
ce = [c['epoch'] for c in coefs]
for name in ('b', 's_r', 'alpha_sum'):
    ax.plot(ce, [c[name] for c in coefs], label=name)
ax.set_title('Learned physical coefficients'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('final coefficients:', {k: coefs[-1][k] for k in ('b', 's_r', 'alpha_sum')})

## Cell 8: Load best checkpoint -> test predictions + metrics

In [ ]:
import numpy as np
import torch
from train_module import load_best

# re-instantiate model with same config, then load best weights
best_model = build_model({'name': 'pinn',
                          'model': {'hidden_dim': 128, 'hidden_blocks': 3, 'dropout': 0.1}})
ckpt = load_best(best_model, result['best_path'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model = best_model.to(device).eval()
print('loaded best epoch=', ckpt['epoch'], 'val_data=', ckpt['val_data'])

# test predictions
preds, targs, x_raws = [], [], []
with torch.no_grad():
    for batch in loaders['test']:
        x = batch['x'].to(device)
        p = best_model(x).cpu().numpy()
        preds.append(p); targs.append(batch['y'].numpy()); x_raws.append(batch['x_raw'].numpy())
pred = np.concatenate(preds); targ = np.concatenate(targs); x_raw = np.concatenate(x_raws)

# simple metrics per axis
def mse(a, b): return float(np.mean((a - b) ** 2))
def mae(a, b): return float(np.mean(np.abs(a - b)))
def rmse(a, b): return float(np.sqrt(mse(a, b)))

print(f"pred shape: {pred.shape}  target shape: {targ.shape}")
print("-- PINN corrected vs GT --")
print(f"  d_x   : MSE={mse(pred[:,0], targ[:,0]):.6e}  MAE={mae(pred[:,0], targ[:,0]):.6e}  RMSE={rmse(pred[:,0], targ[:,0]):.6e}")
print(f"  d_yaw : MSE={mse(pred[:,1], targ[:,1]):.6e}  MAE={mae(pred[:,1], targ[:,1]):.6e}  RMSE={rmse(pred[:,1], targ[:,1]):.6e}")
print("-- Identity baseline (raw odom) vs GT --")
print(f"  d_x   : MSE={mse(x_raw[:,0], targ[:,0]):.6e}  MAE={mae(x_raw[:,0], targ[:,0]):.6e}")
print(f"  d_yaw : MSE={mse(x_raw[:,5], targ[:,1]):.6e}  MAE={mae(x_raw[:,5], targ[:,1]):.6e}")

print('final coefficients :', ckpt.get('coefficients'))

## Cell 9: Full evaluation via eval_module (ATE / FPE / RPE / plots)

Upload `eval_module.py` if it isn't already on the path, then run
`run_full_evaluation` on the test loader. Writes `metrics.json` +
`*_trajectory.png` + `*_error_over_time.png` to `SAVE_DIR`.

In [ ]:
# Ensure eval_module is importable alongside the other modules.
import os, sys
if 'eval_module' not in sys.modules:
    try:
        import eval_module  # noqa: F401
    except ImportError:
        from google.colab import files  # type: ignore
        print('Upload eval_module.py')
        up = files.upload()
        for name, payload in up.items():
            with open(os.path.join(CODE_DIR, name), 'wb') as f:
                f.write(payload)
        sys.path.insert(0, CODE_DIR)
        import eval_module  # noqa: F401

import importlib, eval_module
importlib.reload(eval_module)
print('eval_module loaded:', eval_module.__file__)

In [ ]:
from eval_module import run_full_evaluation

eval_out = run_full_evaluation(
    model=best_model,
    loader=loaders['test'],
    save_dir=SAVE_DIR,
    history=hist,
    final_coeffs=ckpt.get('coefficients'),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    rpe_k=10,
    tag='test',
)
metrics = eval_out['metrics']
print('metrics.json:', metrics['metrics_json'])

## Cell 10: Display evaluation results

In [ ]:
# Summary table
ps = metrics['per_step']; tr = metrics['trajectory']
print('=== Per-step (test) ===')
print(f"  PINN  RMSE d_x   = {ps['pinn_rmse_dx']:.4e} | MAE d_x   = {ps['pinn_mae_dx']:.4e}")
print(f"  Odom  RMSE d_x   = {ps['odom_rmse_dx']:.4e} | MAE d_x   = {ps['odom_mae_dx']:.4e}")
print(f"  PINN  RMSE d_yaw = {ps['pinn_rmse_dyaw']:.4e} | MAE d_yaw = {ps['pinn_mae_dyaw']:.4e}")
print(f"  Odom  RMSE d_yaw = {ps['odom_rmse_dyaw']:.4e} | MAE d_yaw = {ps['odom_mae_dyaw']:.4e}")
print(f"  Improvement RMSE d_x:   {ps['improvement_rmse_dx_pct']:+.2f}%")
print(f"  Improvement RMSE d_yaw: {ps['improvement_rmse_dyaw_pct']:+.2f}%")
print()
print('=== Trajectory (test) ===')
print(f"  Odom-only  ATE={tr['odom_vs_gt']['ATE_RMSE']:.4e}  FPE={tr['odom_vs_gt']['FPE']:.4e}  head_rmse={tr['odom_vs_gt']['heading_err_RMSE']:.4e}")
print(f"  PINN       ATE={tr['pinn_vs_gt']['ATE_RMSE']:.4e}  FPE={tr['pinn_vs_gt']['FPE']:.4e}  head_rmse={tr['pinn_vs_gt']['heading_err_RMSE']:.4e}")
print(f"  ATE reduction: {tr['improvement']['ATE_RMSE_reduction_pct']:+.2f}%  |  FPE reduction: {tr['improvement']['FPE_reduction_pct']:+.2f}%")
print()
print('=== Stationary residual (pred on stationary samples; want near 0) ===')
for k, v in metrics['stationary_residual'].items():
    print(f"  {k}: {v}")
print()
print('=== Coefficients ===')
cs = metrics['coefficients']
print('  final:', cs.get('final'))
print('  convergence (last50 std):')
for k, v in (cs.get('convergence') or {}).items():
    print(f"    {k}: last={v['last']:.5f}  last50_std={v['last50_std']:.3e}  cv={v['last50_cv']:.3e}")
print('  deviation from prior:', cs.get('deviation_from_prior'))

In [ ]:
from IPython.display import Image, display
print('Trajectory plot:')
display(Image(filename=metrics['plots']['trajectory']))
print('Error over time:')
display(Image(filename=metrics['plots']['error_over_time']))

## Cell 11 (optional): Ablation runs

Re-train the model with one physics term disabled at a time (`w_nonholonomic=0`,
`w_stationary=0`, `w_coeff=0`). Evaluate each and compare to the full model.
Disabled by default — uncomment to run (adds ~3× training time).

In [ ]:
# ABLATE = False
# if ABLATE:
#     ablation_runs = {
#         'no_nh':    {'w_nonholonomic': 0.0},
#         'no_stat':  {'w_stationary':   0.0},
#         'no_coeff': {'w_coeff':        0.0},
#     }
#     ablation_metrics = {}
#     for tag, overrides in ablation_runs.items():
#         loss_cfg = {'w_data': 1.0, 'w_stationary': 1.0, 'w_nonholonomic': 0.1,
#                     'w_coeff': 0.01, 'w_magnitude': 1e-3, 
#                     'data_weights': (w_x, w_yaw)}
#         loss_cfg.update(overrides)
#         ab_model = build_model({'name': 'pinn',
#                                 'model': {'hidden_dim': 128, 'hidden_blocks': 3, 'dropout': 0.1}})
#         ab_loss = build_loss({'loss': loss_cfg})
#         ab_dir = f'{SAVE_DIR}_ablation_{tag}'
#         ab_cfg = TrainConfig(epochs=150, early_stop_patience=15, run_name=f'pinn_{tag}',
#                              seed=42, device='auto', amp=True, log_every=25)
#         ab_res = train_api(ab_model, ab_loss, loaders['train'], loaders['val'], ab_cfg, save_dir=ab_dir)
#         load_best(ab_model, ab_res['best_path'])
#         ab_model = ab_model.to(device).eval()
#         out = run_full_evaluation(ab_model, loaders['test'], save_dir=ab_dir,
#                                   history=ab_res['history'],
#                                   final_coeffs=ab_model.coefficient_values(),
#                                   device='cuda' if torch.cuda.is_available() else 'cpu', tag=tag)
#         ablation_metrics[tag] = out['metrics']
#         print(f'[ablation:{tag}] ATE={out["metrics"]["trajectory"]["pinn_vs_gt"]["ATE_RMSE"]:.4e}')
#     import json as _json
#     print(_json.dumps({k: v['trajectory']['pinn_vs_gt'] for k, v in ablation_metrics.items()}, indent=2))